# Fast BGE / fusion evidence swap with the frozen DistilBERT verifier

This evaluates the **existing BM25-trained DistilBERT** on the official 500 AVeriTeC dev claims, once with BGE-small top-three passages and once with BM25+BGE fusion top-three passages. It reuses completed rankings and model weights: **no corpus indexing, embedding, or training**. Run all code cells in order on a Colab GPU. Inference has a progress bar and resumes from Drive after an interruption.

These are **evidence-swap diagnostics**, not matched BGE-trained or fusion-trained systems. The verifier learned from BM25 train evidence, so a better or worse score here does not establish how a verifier trained on the new retriever would perform. Dev labels are used only for scoring. Setup/download time varies; the inference itself should take minutes rather than the hours required to embed the corpus.


In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/10718-ML-In-Practice')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements/colab.txt'], check=True)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['HF_HOME'] = '/content/huggingface_cache'
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, restart, and rerun this cell.')
print('Repo:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import json
from src.utils.io import read_jsonl

DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
DEV_CLAIMS = DATA_ROOT / 'processed/averitec/dev.jsonl'
BM25_DEV = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
PRIOR_ROOT = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16'
BGE_DEV = PRIOR_ROOT / 'dense/dev_bge_small_v15/dev_rankings.jsonl'
FUSION_DEV = PRIOR_ROOT / 'hybrid/dev_rrf60/hybrid_rankings.jsonl'
VERIFIER = PRIOR_ROOT / 'transformer/bm25_top3'
MODEL_DIR = VERIFIER / 'final_model'
OUTPUT_ROOT = DATA_ROOT / 'outputs/initial_ml_solution_dense_fusion_verifiers_v1/frozen_bm25_trained_evidence_swap'

needed = [DEV_CLAIMS, BM25_DEV, BGE_DEV, FUSION_DEV, VERIFIER / 'transformer_run_config.json',
          VERIFIER / 'final_dev_metrics.json', MODEL_DIR / 'config.json']
missing = [str(p) for p in needed if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing completed input(s); check the Drive paths:\n' + '\n'.join(missing))
if not [*MODEL_DIR.glob('*.safetensors'), *MODEL_DIR.glob('pytorch_model*.bin')]:
    raise FileNotFoundError(f'Missing trained DistilBERT weights in {MODEL_DIR}')
dev = read_jsonl(DEV_CLAIMS)
dev_ids = [str(row['claim_id']) for row in dev]
if len(dev_ids) != 500 or len(set(dev_ids)) != 500:
    raise ValueError('Expected exactly 500 distinct official dev claims.')
for path in (BM25_DEV, BGE_DEV, FUSION_DEV):
    rankings = read_jsonl(path)
    ids = [str(row['claim_id']) for row in rankings]
    if len(ids) != 500 or set(ids) != set(dev_ids) or len(set(ids)) != 500:
        raise ValueError(f'Rankings do not cover exactly the same dev claims: {path}')
    if any(not isinstance(row.get('retrieved'), list) for row in rankings):
        raise ValueError(f'Malformed retrieved list in {path}')
print('Validated 500 claims in each ranking file; no retrieval will be recomputed.')
print('Frozen verifier:', MODEL_DIR)
print('Resume/output directory:', OUTPUT_ROOT)


In [ ]:
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scripts.run_evidence_ablation import load_partial_predictions
from scripts.train_tfidf_logreg import attach_retrieved_evidence
from scripts.train_transformer_verifier import compose_text
from src.data.averitec import CANONICAL_LABELS
from src.evaluation.classification import evaluate_predictions
from src.utils.io import write_json, write_jsonl
from src.utils.runs import sha256_file

model_settings = json.loads((VERIFIER / 'transformer_run_config.json').read_text(encoding='utf-8'))
weight_files = sorted((*MODEL_DIR.glob('*.safetensors'), *MODEL_DIR.glob('pytorch_model*.bin')))
weight_hashes = {p.name: sha256_file(p) for p in weight_files}
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True).to('cuda').eval()
if [str(model.config.id2label[i]) for i in range(4)] != list(CANONICAL_LABELS):
    raise ValueError('Saved model label order differs from the evaluator.')

def complete(paths):
    for path in paths:
        if not path.is_file() or path.stat().st_size == 0:
            return False
        if path.suffix == '.json':
            json.loads(path.read_text(encoding='utf-8'))
    return True

def evaluate_with_dev_rankings(name, rankings):
    out = OUTPUT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    predictions_path, metrics_path = out / 'predictions.jsonl', out / 'metrics.json'
    config_path, partial_path = out / 'run_config.json', out / 'predictions.partial.jsonl'
    config = {
        'evaluation_type': 'frozen_BM25_trained_verifier_evidence_swap',
        'dev_claims_sha256': sha256_file(DEV_CLAIMS),
        'dev_rankings_sha256': sha256_file(rankings),
        'model_run_config_sha256': sha256_file(VERIFIER / 'transformer_run_config.json'),
        'model_config_sha256': sha256_file(MODEL_DIR / 'config.json'),
        'model_weights_sha256': weight_hashes,
        'model_dir': str(MODEL_DIR.resolve()),
        'train_evidence': str(DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'),
        'eval_evidence': str(rankings),
        'max_evidence': 3,
    }
    previous = json.loads(config_path.read_text(encoding='utf-8')) if config_path.is_file() else None
    if previous is None and any(p.exists() for p in (predictions_path, metrics_path, partial_path)):
        raise ValueError(f'Existing output lacks its input manifest: {out}. Inspect before reusing it.')
    if previous is not None and previous != config:
        raise ValueError(f'Inputs changed for {out}; use a new output directory.')
    if complete((predictions_path, metrics_path)):
        print(f'{name}: complete; reusing saved result.')
        return json.loads(metrics_path.read_text(encoding='utf-8'))
    write_json(config, config_path)
    records = attach_retrieved_evidence(dev, rankings)
    completed = load_partial_predictions(partial_path, set(dev_ids))
    remaining = [row for row in records if str(row['claim_id']) not in completed]
    with tqdm(total=500, initial=len(completed), desc=f'Frozen DistilBERT / {name}', unit='claim') as bar:
        for start in range(0, len(remaining), 16):
            batch = remaining[start:start + 16]
            texts = [compose_text(row, tokenizer, int(model_settings['claim_token_budget']),
                                  int(model_settings['evidence_token_budget_per_passage'])) for row in batch]
            encoded = tokenizer(texts, padding=True, truncation=True,
                                max_length=int(model_settings['max_length']), return_tensors='pt')
            encoded = {key: value.to('cuda') for key, value in encoded.items()}
            with torch.inference_mode():
                probabilities = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu().tolist()
            pending = []
            for row, values in zip(batch, probabilities, strict=True):
                claim_id = str(row['claim_id'])
                evidence = (row.get('retrieved_evidence') or [])[:3]
                prediction = {
                    'claim_id': claim_id, 'true_label': str(row['label']),
                    'predicted_label': CANONICAL_LABELS[max(range(4), key=values.__getitem__)],
                    'probabilities': {label: float(values[i]) for i, label in enumerate(CANONICAL_LABELS)},
                    'evidence_item_count': len(evidence),
                    'evidence_passage_ids': [str(item['passage_id']) for item in evidence],
                }
                completed[claim_id] = prediction
                pending.append(prediction)
            with partial_path.open('a', encoding='utf-8', newline='\n') as handle:
                for row in pending:
                    handle.write(json.dumps(row, ensure_ascii=False) + '\n')
                handle.flush()
            bar.update(len(pending))
    if len(completed) != 500:
        raise RuntimeError(f'Only {len(completed)} of 500 claims completed.')
    ordered = [completed[claim_id] for claim_id in dev_ids]
    temporary = predictions_path.with_name(predictions_path.name + '.tmp')
    write_jsonl(ordered, temporary)
    temporary.replace(predictions_path)
    report = {**evaluate_predictions(ordered), **config,
              'claim_count': 500, 'matched_train_eval_evidence_pipeline': False}
    temporary_metrics = metrics_path.with_name(metrics_path.name + '.tmp')
    write_json(report, temporary_metrics)
    temporary_metrics.replace(metrics_path)
    return report

RESULTS = {}
for name, path in (('dense', BGE_DEV), ('fusion', FUSION_DEV)):
    RESULTS[name] = evaluate_with_dev_rankings(name, path)
    errors = RESULTS[name]['decision_error_rates']
    print(name, 'macro-F1=', round(RESULTS[name]['macro_f1'], 4),
          'accuracy=', round(RESULTS[name]['accuracy'], 4),
          'FR=', errors['false_reassurance_count'], 'FA=', errors['false_alarm_count'])
del model
torch.cuda.empty_cache()


In [ ]:
baseline = json.loads((VERIFIER / 'final_dev_metrics.json').read_text(encoding='utf-8'))
reports = [('BM25 top three (matched training)', baseline),
           ('BGE-small top three (evidence swap)', RESULTS['dense']),
           ('BM25+BGE fusion top three (evidence swap)', RESULTS['fusion'])]
print('All verdict rows: same 500 official dev claims; FR and FA are counts out of 500.')
print(f'{"Evidence at inference":46} {"Macro-F1":>9} {"Accuracy":>9} {"FR":>5} {"FA":>5}')
for name, report in reports:
    errors = report['decision_error_rates']
    print(f'{name:46} {report["macro_f1"]:9.4f} {report["accuracy"]:9.4f} '          f'{errors["false_reassurance_count"]:5d} {errors["false_alarm_count"]:5d}')
print('\nEvidence-swap results are NOT matched BGE/fusion-trained verifiers.')
print('Saved under:', OUTPUT_ROOT)
